In [ ]:
!pip install pm4py

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pm4py
import pandas as pd
log = pm4py.read_xes ('event_log.xes')
log ['time:timestamp'] = pd.to_datetime(log ['time:timestamp'])

print("Total events:", len(log))
print("Total Case:", log['case_id'].nunique())
print("Date range:", log["time:timestamp"].min(), "to", log["time:timestamp"].max())
print("\nActivity frequency:")
print(log['activity'].value_counts())

In [ ]:
print("Number of unique activities:", log['activity'].nunique())
print("Number of resources(Staff):", log['resource'].nunique())
print("Resources:", log['resource'].nunique())

# Case duration - how long does each order take start to finish?
case_duration = log.groupby('case_id')['time:timestamp'].agg(['min', 'max'])
case_duration['duration'] = case_duration['max'] - case_duration['min']
print("Average case duration:", case_duration['duration'].mean())
print("Longest case duration:", case_duration['duration'].max())
print("Shortest case duration:", case_duration['duration'].min())

In [ ]:
log['year'] = log['time:timestamp'].dt.year
log_2030 = log[log['year'] == 2030].copy()
log_2030['case_id'] = log_2030['case_id'].astype(str)

print("Cases in 2030:", log_2030['case_id'].nunique())
print("Events in 2030:", len(log_2030))

In [ ]:
# Filter 2: keep only cases that reached 'Order_is_served' (completed orders)
served_cases = log_2030[log_2030['activity'] == 'Order_is_served']['case_id'].unique()
log_2030_completed = log_2030[log_2030['case_id'].isin(served_cases)].copy()

print("Completed Cases:", log_2030_completed ['case_id'].nunique())
print("Completed Events:", len(log_2030_completed))

In [ ]:
# Force Python to convert the Case ID numbers into text strings
log_2030_completed['case_id'] = log_2030_completed['case_id'].astype(str)

event_log = pm4py.convert_to_event_log(log_2030_completed, case_id_key='case_id', activity_key='activity', timestamp_key='time:timestamp')

In [ ]:
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(event_log)
pm4py.view_petri_net(net, initial_marking, final_marking)

In [ ]:
bpmn_model = pm4py.discover_bpmn_inductive(event_log)
pm4py.view_bpmn(bpmn_model)

In [ ]:
dfg, start_activities, end_activities = pm4py.discover_dfg(event_log)
pm4py.view_dfg(dfg, start_activities, end_activities)

In [ ]:
performance_dfg, start_activities, end_activities = pm4py.discover_performance_dfg(event_log)
pm4py.view_performance_dfg(performance_dfg, start_activities, end_activities)

In [ ]:
from pm4py.algo.discovery.temporal_profile import algorithm as temporal_profile_discovery
temporal_profile = temporal_profile_discovery.apply(event_log)

# Print mean and std dev for every pair of consecutive activities
for pair, (mean, std) in temporal_profile.items():
      print(f"{pair[0]} -> {pair[1]}: mean = {mean:.2f} sec, std dev = {std:.2f} sec")

In [ ]:
import pandas as pd

rows = []
for pair, (mean, std) in temporal_profile.items():
      rows.append ({
            'From': pair[0],
            'To': pair[1],
            'Mean (hours)': round (mean/3600, 2),
            'Std Dev (hours)': round (std/3600, 2)
      })

temporal_df = pd.DataFrame(rows).sort_values('Mean (hours)', ascending=False)
print(temporal_df)